# Studi Kasus dan Latihan Praktikum 2

**Mata kuliah:** Praktikum Big Data  
**Nama:** Arya Pratama Hendri  
**NIM:** 2411533008  
**Kelas:** B


## P. Studi Kasus

Platform marketplace mencatat 515 baris pada data mentah, sedangkan tim Finance memakai 490 baris setelah proses preprocessing.

### 1. Mengapa jumlah data tim IT dan tim Finance berbeda?

Kedua tim memakai data dari tahap yang berbeda. Tim IT melihat 515 baris data mentah. Di dalam jumlah tersebut masih ada 15 baris duplicate dan 10 baris yang tidak memiliki `customer_name` atau `payment_method`.

Proses cleaning menghapus 15 salinan transaksi, sehingga jumlahnya menjadi 500 baris. Setelah itu, 10 baris dengan data wajib yang kosong ikut dibuang. Tim Finance akhirnya memakai 490 baris yang lolos pemeriksaan.

### 2. Apakah 490 baris lebih benar dibandingkan 515 baris jika dikaitkan dengan Veracity?

Untuk menghitung laporan transaksi, 490 baris lebih dapat dipercaya. Setiap baris hanya tercatat satu kali dan memiliki informasi inti yang lengkap. Jika tim Finance memakai 515 baris, transaksi duplicate dapat membuat total penjualan terlihat lebih besar dari hasil yang seharusnya.

Angka 490 tetap berasal dari dataset simulasi. Proses cleaning meningkatkan Veracity dengan mengurangi kesalahan yang terlihat pada data, tetapi tidak mengubah dataset sintetis menjadi data transaksi nyata.

### 3. Mengapa missing value pada rating dibiarkan kosong?

Pembeli tidak wajib memberi rating. Nilai kosong pada kolom tersebut berarti tidak ada penilaian yang masuk. Mengisinya dengan angka tebakan akan menambahkan pendapat yang tidak pernah diberikan pembeli dan dapat mengubah nilai rata-rata.

Rata-rata rating sebaiknya dihitung dari transaksi yang memiliki rating. Tim Finance juga perlu menerima jumlah atau persentase transaksi yang memberi rating agar mereka memahami cakupan hasil rata-ratanya. Dalam pandas, `mean()` mengabaikan nilai `NaN` saat menghitung rata-rata.

## Q. Latihan

### Persiapan Latihan

Library berikut diperlukan agar Latihan 1 dapat dijalankan tanpa bergantung pada cell di notebook lain.

In [1]:
!pip install faker -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 56.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from faker import Faker
import random

### Latihan 1 - Membandingkan SEED 7 dan SEED 42

Kode berikut membungkus proses pembuatan dan pembersihan data ke dalam fungsi `jalankan_pipeline()`. Parameter `seed` menentukan pola angka acak yang digunakan. Fungsi mengembalikan data mentah dan data bersih supaya jumlah baris keduanya dapat dibandingkan.

In [3]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan

    nilai = str(x).strip().replace("Rp", "")

    if nilai.endswith(".0") and nilai[:-2].isdigit():
        nilai = nilai[:-2]
    else:
        nilai = nilai.replace(".", "")

    nilai = nilai.replace(",", ".")

    try:
        return float(nilai)
    except ValueError:
        return np.nan


def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT


def jalankan_pipeline(seed):
    np.random.seed(seed)
    random.seed(seed)

    fake = Faker("id_ID")
    fake.seed_instance(seed)

    jumlah_data = 500
    kategori_produk = [
        "Elektronik",
        "Fashion",
        "Kesehatan",
        "Rumah Tangga",
        "Olahraga",
        "Buku",
    ]
    metode_bayar = [
        "Transfer Bank",
        "E-Wallet",
        "COD",
        "Kartu Kredit",
    ]

    rows = []

    for i in range(1, jumlah_data + 1):
        trx_id = f"TRX{i:05d}"
        nama_pelanggan = fake.name()
        produk = fake.word().capitalize() + " " + random.choice(
            ["Pro", "Lite", "Max", "Basic", ""]
        )
        kategori = random.choice(kategori_produk)
        harga_dasar = random.choice(
            [15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000]
        )
        qty = random.randint(1, 5)

        harga_variants = [
            str(harga_dasar),
            f"Rp{harga_dasar:,}".replace(",", "."),
            f"{harga_dasar}.0",
            f" {harga_dasar} ",
        ]
        harga = random.choice(harga_variants)

        tgl = fake.date_between(start_date="-90d", end_date="today")
        tgl_variants = [
            tgl.strftime("%Y-%m-%d"),
            tgl.strftime("%d/%m/%Y"),
            tgl.strftime("%d-%m-%Y"),
        ]
        tanggal = random.choice(tgl_variants)

        metode = random.choice(metode_bayar)
        if random.random() < 0.3:
            metode = metode.lower()
        if random.random() < 0.2:
            kategori = kategori.upper() + " "

        rows.append(
            {
                "transaction_id": trx_id,
                "customer_name": nama_pelanggan,
                "product_name": produk.strip(),
                "category": kategori,
                "price": harga,
                "quantity": qty,
                "payment_method": metode,
                "transaction_date": tanggal,
                "shipping_city": fake.city(),
                "rating": random.choice([1, 2, 3, 4, 5, None, None]),
            }
        )

    df_mentah = pd.DataFrame(rows)

    for col, frac in [
        ("customer_name", 0.02),
        ("shipping_city", 0.03),
        ("payment_method", 0.015),
    ]:
        idx = df_mentah.sample(frac=frac, random_state=seed).index
        df_mentah.loc[idx, col] = np.nan

    duplicate = df_mentah.sample(n=15, random_state=seed)
    df_mentah = pd.concat([df_mentah, duplicate], ignore_index=True)
    df_mentah = df_mentah.sample(frac=1, random_state=seed).reset_index(drop=True)

    df_bersih = df_mentah.drop_duplicates().copy()
    df_bersih = df_bersih.dropna(
        subset=["customer_name", "payment_method"]
    ).copy()
    df_bersih["shipping_city"] = df_bersih["shipping_city"].fillna(
        "Tidak Diketahui"
    )

    for col in ["category", "payment_method", "shipping_city"]:
        df_bersih[col] = (
            df_bersih[col].astype("string").str.strip().str.title()
        )

    df_bersih["payment_method"] = df_bersih["payment_method"].replace(
        {"Cod": "COD"}
    )
    df_bersih["price"] = df_bersih["price"].apply(bersihkan_harga)
    df_bersih["transaction_date"] = (
        df_bersih["transaction_date"]
        .apply(parse_tanggal)
        .dt.strftime("%Y-%m-%d")
    )
    df_bersih["quantity"] = df_bersih["quantity"].astype(int)
    df_bersih["price"] = df_bersih["price"].astype(float)

    return df_mentah, df_bersih


df_mentah_7, df_bersih_7 = jalankan_pipeline(7)
df_mentah_42, df_bersih_42 = jalankan_pipeline(42)

print(
    "SEED = 7  :",
    len(df_mentah_7),
    "baris mentah dan",
    len(df_bersih_7),
    "baris bersih",
)
print(
    "SEED = 42 :",
    len(df_mentah_42),
    "baris mentah dan",
    len(df_bersih_42),
    "baris bersih",
)

SEED = 7  : 515 baris mentah dan 490 baris bersih
SEED = 42 : 515 baris mentah dan 490 baris bersih


**Jawaban Latihan 1:**

SEED 7 dan SEED 42 menghasilkan jumlah yang sama, yaitu 515 baris mentah dan 490 baris bersih. Program selalu membuat 500 transaksi dan menambahkan 15 duplicate. Program juga memakai persentase missing value yang tetap, sehingga jumlah baris yang dibuang tidak berubah.

Perbedaan seed memengaruhi isi data dan baris mana yang terpilih. Nama pelanggan, kota, kategori, serta posisi missing value dapat berbeda walaupun jumlah akhirnya sama.

### Latihan 2 - Memeriksa Harga yang Tidak Valid

Kolom `is_valid_price` berisi `True` jika harga lebih besar dari nol. Nilai `False` menandakan harga nol, negatif, atau gagal memenuhi syarat pemeriksaan.

In [4]:
df_bersih_42["is_valid_price"] = df_bersih_42["price"] > 0

jumlah_tidak_valid = (~df_bersih_42["is_valid_price"]).sum()

print("Jumlah harga tidak valid:", jumlah_tidak_valid)
print("\nHasil pemeriksaan:")
print(df_bersih_42["is_valid_price"].value_counts())

Jumlah harga tidak valid: 0

Hasil pemeriksaan:
is_valid_price
True    490
Name: count, dtype: int64


**Jawaban Latihan 2:**

Hasil pemeriksaan menunjukkan 0 harga tidak valid. Seluruh 490 transaksi memiliki harga lebih besar dari nol, sehingga nilai pada kolom `is_valid_price` semuanya `True`.

### Latihan 3 - Menghitung Transaksi per Kategori

`value_counts()` menghitung jumlah kemunculan setiap kategori pada dataset bersih. Hasilnya diurutkan dari kategori dengan transaksi terbanyak.

In [5]:
jumlah_per_kategori = df_bersih_42["category"].value_counts()

print("Jumlah transaksi per kategori:")
print(jumlah_per_kategori)

Jumlah transaksi per kategori:
category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


**Jawaban Latihan 3:**

Hasil `value_counts()` menunjukkan penyebaran 490 transaksi pada enam kategori. Jumlah tiap kategori berasal dari data acak dengan SEED 42, sehingga hasilnya dapat diperoleh kembali saat pipeline dijalankan dengan seed yang sama.